# Study CAX M1 misalignment effects on the X-ray beam

```2026/07/10```

In [ ]:
import os
import sys

sys.path.append(os.path.expanduser("~/repos/cax-scripts"))
# sys.path.append(os.path.expanduser("~/repos/cax-control"))

from shadowpy.cax_simulation import CAXSim
from caxscripts.image_statistics import Histogram2DAnalyzer
import numpy as np
import matplotlib.pyplot as plt
from shadowpy.utils import save_image

In [ ]:
cax = CAXSim(total_rays=100000)

In [ ]:
def _perform_plot_caustic(cax: CAXSim):
    dist, fwhm_x, fwhm_y = \
        cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam.duplicate(),
                                    (-250, 250),
                                    n_points=100, 
                                    max_workers=10)

    astig = dist[np.nanargmin(fwhm_x)] - dist[np.nanargmin(fwhm_y)]

    plt.figure(figsize=(12, 6))
    plt.plot(dist, 1000*fwhm_x, marker='o', markersize=5, label="FWHM X")
    plt.plot(dist, 1000*fwhm_y, marker='o', markersize=5, label="FWHM Y")
    plt.vlines(dist[np.argmin(fwhm_x)], 0, 1000*np.min(fwhm_x), color='blue', linestyle='--', label=f"Min FWHM X: {1000*np.min(fwhm_x):.2f} μm at {dist[np.argmin(fwhm_x)]:.2f} mm")
    plt.vlines(dist[np.argmin(fwhm_y)], 0, 1000*np.min(fwhm_y), color='orange', linestyle='--', label=f"Min FWHM Y: {1000*np.min(fwhm_y):.2f} μm at {dist[np.argmin(fwhm_y)]:.2f} mm")
    plt.xlabel('Distance from screen (mm)')
    plt.ylabel('FWHM (μm)')
    plt.title(f'Caustic of the beam: astigmatism of {astig:.2f} mm')
    plt.legend()
    plt.grid()
    plt.show()

    return astig

In [ ]:
# def _perform_plot_caustic(cax: CAXSim):
#     dist, fwhm_x, fwhm_y = \
#         cax.dvf_B1.simple_caustic(cax.dvf_B1.beam.duplicate(),
#                                     (-250, 250),
#                                     n_points=50)

#     astig = dist[np.nanargmin(fwhm_x)] - dist[np.nanargmin(fwhm_y)]

#     plt.figure(figsize=(12, 6))
#     plt.plot(dist, 1000*fwhm_x, marker='o', markersize=5, label="FWHM X")
#     plt.plot(dist, 1000*fwhm_y, marker='o', markersize=5, label="FWHM Y")
#     plt.vlines(dist[np.argmin(fwhm_x)], 0, 1000*np.min(fwhm_x), color='blue', linestyle='--', label=f"Min FWHM X: {1000*np.min(fwhm_x):.2f} μm at {dist[np.argmin(fwhm_x)]:.2f} mm")
#     plt.vlines(dist[np.argmin(fwhm_y)], 0, 1000*np.min(fwhm_y), color='orange', linestyle='--', label=f"Min FWHM Y: {1000*np.min(fwhm_y):.2f} μm at {dist[np.argmin(fwhm_y)]:.2f} mm")
#     plt.xlabel('Distance from screen (mm)')
#     plt.ylabel('FWHM (μm)')
#     plt.title(f'Caustic of the beam: astigmatism of {astig:.2f} mm')
#     plt.legend()
#     plt.grid()
#     plt.show()

#     return astig

In [ ]:
_perform_plot_caustic(cax)

In [ ]:
def _astigmatism_acquisition(cax: CAXSim):

    # Initial search

    low_lim = -1000
    hgh_lim =  1000

    done = False

    while not done:
        distances, fwhm_x, fwhm_y = \
            cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam.duplicate(),
                                        s_range=(low_lim, hgh_lim),
                                        n_points=50, 
                                        max_workers=10)
        # prior focal lengths
        fx = distances[np.nanargmin(fwhm_x)]
        fy = distances[np.nanargmin(fwhm_y)]

        print(fx, fy)

        done = True

        # check if the search was enought otherwise repeat
        if low_lim in [fx, fy]:
            low_lim -= 100
            done = False

        if hgh_lim in [fx, fy]:
            hgh_lim += 100
            done = False

    # Finer search
    distances, fwhm_x, fwhm_y = \
    cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam.duplicate(),
                                s_range=(np.min([fx, fy])-50, np.max([fx, fy])+50),
                                n_points=100, 
                                max_workers=10)

    # actual focal lengths
    fx = distances[np.nanargmin(fwhm_x)]
    fy = distances[np.nanargmin(fwhm_y)]
    print(fx, fy)

    f = np.mean([fx, fy])
    retr_beam = cax.dvf_B1.beam.duplicate()
    retr_beam.retrace(f)
    ana = save_image(cax.dvf_B1, retr_beam)
    astig = fy - fx

    return distances, fwhm_x, fwhm_y, fx, fy, f, astig, ana 

In [ ]:
def image_peak(image: Histogram2DAnalyzer):
    
    
    # print("image_peak: hprm_momenta type -- ", type(image.hprm_momenta))
    total = image.img.sum()

    fx = image.hprm_fitting['fwhmx']
    fy = image.hprm_fitting['fwhmy']

    I = total/(fx*fy*1000**2)
    return I

def acquisition_func(cax: CAXSim):
    
    # What observables do I want? centroid (x, y), fwhm (x, y), peak (roi avg), astig
    distances, fwhm_x, fwhm_y, fx, fy, f, astig, ana = _astigmatism_acquisition(cax)
    if not ana.beam_visible:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
    peak = image_peak(ana)
    cx = ana.hprm_fitting['mux']
    cy = ana.hprm_fitting['muy']

    fwhmx = ana.hprm_fitting['fwhmx']
    fwhmy = ana.hprm_fitting['fwhmy']

    return cx, cy, fwhmx, fwhmy, peak, astig


In [ ]:
def motorscans(cax: CAXSim, tx_vals: np.ndarray, ry_vals: np.ndarray):

        
    shape = (len(tx_vals), len(ry_vals))
    
    cx_array    = np.full(shape, np.nan)
    cy_array    = np.full(shape, np.nan)
    fwhmx_array = np.full(shape, np.nan)
    fwhmy_array = np.full(shape, np.nan)
    peak_array  = np.full(shape, np.nan)
    astig_array = np.full(shape, np.nan)


    for i, tx in enumerate(tx_vals):
        cax.mirror.tx = tx

        for j, ry in enumerate(ry_vals):
            cax.mirror.ry = ry

            cx, cy, fwhmx, fwhmy, peak, astig = acquisition_func(cax)

            cx_array[i, j] = cx
            cy_array[i, j] = cy
            fwhmx_array[i, j] = fwhmx
            fwhmy_array[i, j] = fwhmy
            peak_array[i, j] = peak
            astig_array[i, j] = astig

            print(f"Iteration {i*len(tx_vals)+j:4d}/{len(tx_vals)*len(ry_vals):4d}")
    return {
        "cx": cx_array,
        "cy": cy_array,
        "fwhmx": fwhmx_array,
        "fwhmy": fwhmy_array,
        "peak": peak_array,
        "astig": astig_array,
    }


In [ ]:
# tx_vals = np.arange(-1, 1.05, 0.05)
# ry_vals = np.arange(-.1, .105, 0.005)


tx_vals = np.arange(-1, 1.1, 0.1)
ry_vals = np.arange(-.1, .11, 0.01)

In [ ]:
data = motorscans(cax, tx_vals, ry_vals)

In [ ]:
data.keys()

In [ ]:
def plot_scan(tx_vals,
              ry_vals,
              observable,
              title="",
              cmap="viridis",
              ax=None):

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    mesh = ax.pcolormesh(
        tx_vals,
        ry_vals,
        observable.T,   # transpose because observable[i,j]
        shading="auto",
        cmap=cmap
    )

    cbar = plt.colorbar(mesh, ax=ax)
    cbar.set_label(title)

    ax.set_xlabel("tx")
    ax.set_ylabel("ry")
    ax.set_title(title)

    return ax


In [ ]:
data['peak']

In [ ]:
plot_scan(tx_vals, ry_vals, data['fwhmx'], "")
plt.show()

In [ ]:
def plot_all_observables(
    tx_vals,
    ry_vals,
    cx,
    cy,
    fwhmx,
    fwhmy,
    peak,
    astig
):

    observables = [
        (cx, "Centroid X"),
        (fwhmx, "FWHM X"),
        (peak, "Peak Intensity"),
        (cy, "Centroid Y"),
        (fwhmy, "FWHM Y"),
        (astig, "Astigmatism"),
    ]

    fig, axes = plt.subplots(
        2, 3,
        figsize=(15, 9),
        constrained_layout=True
    )

    for ax, (data, title) in zip(axes.flat, observables):

        mesh = ax.pcolormesh(
            tx_vals,
            ry_vals,
            data.T,
            shading="auto",        
        )

        ax.set_title(title)
        ax.set_xlabel("tx")
        ax.set_ylabel("ry")

        plt.colorbar(mesh, ax=ax)

    return fig, axes


In [ ]:
cax.mirror.offset, cax.mirror.tilt

In [ ]:
cax.mirror.tx = 0; cax.mirror.ry = 0

In [ ]:
# image_peak(cax.dvf_B1.image)
plt.imshow(cax.dvf_B1.image.img)

In [ ]:
objective = np.sqrt(data['fwhmx']**2 + data['fwhmy']**2 + np.abs(data['astig'])**2)

In [ ]:
plot_scan(tx_vals, ry_vals, objective, "$\\sqrt{\\mathrm{FWHM}_x^2 + \\mathrm{FWHM}_x^2 + |d|^2}$")

In [ ]:
old_data = cax.dvf_B1.image.img.copy()

In [ ]:
new_data = cax.dvf_B1.image.img.copy()

In [ ]:
np.array_equal(old_data, new_data)

In [ ]:
image_peak(cax.dvf_B1.image)

In [ ]:
data['peak']

In [ ]:

fig, axes = plot_all_observables(
    tx_vals,
    ry_vals,
    data['cx'],
    data['cy'],
    data['fwhmx'],
    data['fwhmy'],
    data['peak'],
    np.abs(data['astig'])
)

plt.show()


In [ ]:
cax.mirror.tx = 0
cax.mirror.ry = -0.1

In [ ]:
distances, fwhm_x, fwhm_y, fx, fy, f, astig, ana = _astigmatism_acquisition(cax)

In [ ]:
fy

In [ ]:
fwhm_x[np.nanargmin(fwhm_x)]
distances[np.nanargmin(fwhm_x)]

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(distances, 1000*fwhm_x, marker='o', markersize=5, label="FWHM X")
plt.plot(distances, 1000*fwhm_y, marker='o', markersize=5, label="FWHM Y")
plt.vlines(distances[np.nanargmin(fwhm_x)], 0, 1000*np.nanmin(fwhm_x), color='blue', linestyle='--', label=f"Min FWHM X: {1000*np.min(fwhm_x):.2f} μm at {distances[np.argmin(fwhm_x)]:.2f} mm")
plt.vlines(distances[np.nanargmin(fwhm_y)], 0, 1000*np.nanmin(fwhm_y), color='orange', linestyle='--', label=f"Min FWHM Y: {1000*np.min(fwhm_y):.2f} μm at {distances[np.argmin(fwhm_y)]:.2f} mm")
plt.vlines(f, 0, 50, color='black', label=f"Midpoint of the foci: {f:.2f} mm")
plt.xlabel('Distance from screen (mm)')
plt.ylabel('FWHM (μm)')
plt.title(f'Caustic of the beam: astigmatism of {astig:.2f} mm')
plt.legend()
plt.grid()
plt.show()

In [ ]:
np.min(data['fwhmy'])/2.355

In [ ]:
np.deg2rad(0.01)

In [ ]:
cax.mirror.ry

In [ ]:
cax.mirror.ry = 0.01

In [ ]:
ana.print_stats(ana.hprm_fitting)

In [ ]:
# cax.mirror.ry = -11.e-4
# cax.mirror.ry
# cax.mirror.tx = 10.e-3
# cax.mirror.rx = 2.e-3

cax.mirror.tx = 0.1
cax.mirror.ry = 0.01

In [ ]:
_perform_plot_caustic(cax)

In [ ]:
plt.imshow(cax.mirror.image.img.T)

In [ ]:
cax.dvf_B1.image.plot()

In [ ]:
cax.dvf_B1.image.print_stats(hprm=cax.dvf_B1.image.hprm_fitting)

In [ ]:
cax.dvf_B1.image.print_stats(hprm=cax.dvf_B1.image.hprm_fitting)

In [ ]:
cax.mirror.tx
np.rad2deg(cax.mirror.rx)

In [ ]:
cax.mirror.offset

In [ ]:
cax.mirror.rx = 0.1

In [ ]:
cax.mirror.shadow_oe.Z_ROT

In [ ]:
np.deg2rad(0.1)

In [ ]:
cax.mirror.tilt

In [ ]:
cax.mirror.image.plot()

In [ ]:
def CAX_SVD(cax: CAXSim, dT: float = 0.01, dR: float = 0.01):
    """
    Perform SVD analysis on the CAX simulation to determine the sensitivity of the system to misalignments.

    Parameters:
    cax (CAXSim): The CAX simulation object.
    dT (float): The translation step size in mm.
    dR (float): The rotation step size in degrees.

    Returns:
    None
    """
    
    T0 = cax.mirror.offset
    R0 = cax.mirror.tilt

    observables = ['fwhmx', 'fwhmy', 'peak', 'astigmatism']
    knobs = ['tx', 'ty', 'rx', 'ry', 'rz']

    fwhm_x0 = cax.dvf_B1.image.hprm_fitting['fwhmx']
    fwhm_y0 = cax.dvf_B1.image.hprm_fitting['fwhmy']
    peak0 = cax.dvf_B1.image.img.max()

    caustic_offsets = np.linspace(-150, 100, 100)

    dist, fwhm_x, fwhm_y = \
        cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam,
                                    (-250, 250),
                                    n_points=100, 
                                    max_workers=10)

    astigmatism0 = dist[np.argmin(fwhm_x)] - dist[np.argmin(fwhm_y)]

    # Jacobian matrix
    J = np.zeros((len(observables), 5))  # 5 misalignment parameters: tx, ty, rx, ry, rz

    for i, obs in enumerate(observables):
        # Perturb each misalignment parameter and compute the change in the observable
        for j, param in enumerate(knobs):
            # Save original value
            original_value = getattr(cax.mirror, param)

            # Perturb the parameter
            if param in ['tx', 'ty']:
                setattr(cax.mirror, param, original_value + dT)
                delta = dT
            else:
                setattr(cax.mirror, param, np.rad2deg(original_value) + dR)
                delta = dR

            # Compute the observable after perturbation
            if obs == 'fwhmx':
                J[i, j] = (cax.dvf_B1.image.hprm_fitting['fwhmx'] - fwhm_x0)/delta
            elif obs == 'fwhmy':
                J[i, j] = (cax.dvf_B1.image.hprm_fitting['fwhmy'] - fwhm_y0)/delta
            elif obs == 'peak':
                J[i, j] = (cax.dvf_B1.image.img.max() - peak0)/delta
            elif obs == 'astigmatism':
                dist, fwhm_x, fwhm_y = \
                    cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam,
                                    (-250, 250),
                                    n_points=100, 
                                    max_workers=10)

                astig = dist[np.argmin(fwhm_x)] - dist[np.argmin(fwhm_y)]
                J[i, j] = (astig - astigmatism0)/delta
            # Restore original value
            setattr(cax.mirror, param, original_value)

    print(f"Jacobian matrix: {J}")

    U, S, Vt = np.linalg.svd(J, full_matrices=False)

    return J, U, S, Vt


In [ ]:
cax = CAXSim(total_rays=500000)

In [ ]:
J, U, S, Vt = CAX_SVD(cax, dT=0.01, dR=0.01)

In [ ]:
# jacobian

for i, obs in enumerate(['fwhmx', 'fwhmy', 'peak', 'astigmatism']):
    print(f"Observable: {obs}")
    for j, param in enumerate(['tx', 'ty', 'rx', 'ry', 'rz']):
        print(f"  Sensitivity to {param}: {J[i, j]:.4f}")

In [ ]:
# Mode 0:

print(f"Singular value associated with mode 0: {S[0]}")

print(f"Combination of knobs to turn: {np.round(Vt[0], 3)}")
print("*Order of knobs: tx, ty, rx, ry, rz")

print(f"Observable effect by exciting mode 0: {np.round(U.T[0], 3)}")
print("*Order of observables: fwhm_x, fwhm_y, peak, astigmatism")

print("=="*40)
print("Conclusion: turning RY has a strong, isolated effect on astigmatism")
print("=="*40)


# Mode 1:

print(f"Singular value associated with mode 1: {S[1]}")

print(f"Combination of knobs to turn: {np.round(Vt[1], 3)}")
print("*Order of knobs: tx, ty, rx, ry, rz")

print(f"Observable effect by exciting mode 1: {np.round(U.T[1], 3)}")
print("*Order of observables: fwhm_x, fwhm_y, peak, astigmatism")

print("=="*40)
print("Conclusion: turning RX decreases the measured peak")
print("=="*40)


# Mode 2:

print(f"Singular value associated with mode 2: {S[2]}")

print(f"Combination of knobs to turn: {np.round(Vt[2], 3)}")
print("*Order of knobs: tx, ty, rx, ry, rz")

print(f"Observable effect by exciting mode 2: {np.round(U.T[2], 3)}")
print("*Order of observables: fwhm_x, fwhm_y, peak, astigmatism")

print("=="*40)
# print("Conclusion: turning RY has a strong, isolated effect on astigmatism")
print("=="*40)

# Mode 0:

print(f"Singular value associated with mode 3: {S[3]}")

print(f"Combination of knobs to turn: {np.round(Vt[3], 3)}")
print("*Order of knobs: tx, ty, rx, ry, rz")

print(f"Observable effect by exciting mode 3: {np.round(U.T[3], 3)}")
print("*Order of observables: fwhm_x, fwhm_y, peak, astigmatism")

print("=="*40)
print("Conclusion: turning RY has a strong, isolated effect on astigmatism")
print("=="*40)









In [ ]:
desired_observables = np.array([46.85, 27.08, 150, 0])

In [ ]:
J_inv = Vt.T @ np.diag(1/S) @ U.T

In [ ]:
J_inv @ desired_observables

In [ ]:
cax.dvf_B1.image.plot()